# dbt Learning — Student Progress Checker

Queries `SANDBOX_DBT_TRAINING` and infers how far each student has progressed through the lessons based on what objects exist in their user-prefixed schemas.

**Schema naming convention:** each student writes to `{PREFIX}_RAW`, `{PREFIX}_STAGING`, `{PREFIX}_INTERMEDIATE`, `{PREFIX}_MARTS`  
**Prefix format:** `JSNOW` from username `jon.snow@company.com`

**Detectable lessons:**
| Lesson | Evidence |
|--------|----------|
| L1 — Project Setup | `{P}_RAW`: customers, orders · `{P}_STAGING`: stg_raw__customers |
| L3 — Staging Layer | All 5 seeds in `{P}_RAW` · All 5 `stg_raw__*` views in `{P}_STAGING` |
| L4 — Intermediate & Marts | 3 `int_*` views in `{P}_INTERMEDIATE` · dim_customers + fct_orders in `{P}_MARTS` |
| L7 — Snapshots | `{P}_SNAPSHOTS.snap_orders` |
| L12 — Production Patterns | `fct_orders_incremental` in `{P}_MARTS` |

> L5 (testing), L6 (dbt_project.yml), L8 (macros), L9 (docs), L10 (dbt build) leave no persistent schema artifacts and cannot be detected from the database.

---
**Run:** Open in Snowflake Workspace and click **Run All**. Change `DATABASE` in the config cell if needed.

In [ ]:
import pandas as pd
from IPython.display import display

# ── Configuration ─────────────────────────────────────────────────────────────
DATABASE = "SANDBOX_DBT_TRAINING"
# ──────────────────────────────────────────────────────────────────────────────
print(f"Database: {DATABASE}")

In [ ]:
%%sql -r raw_objects
SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE
FROM {{DATABASE}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA RLIKE '^[A-Z0-9]+_(RAW|STAGING|INTERMEDIATE|MARTS|SNAPSHOTS)$'
ORDER BY TABLE_SCHEMA, TABLE_NAME

In [ ]:
df_all = raw_objects.copy()
print(f"Found {len(df_all)} objects across {df_all['TABLE_SCHEMA'].nunique()} schemas")

SCHEMA_SUFFIXES = ["_RAW", "_STAGING", "_INTERMEDIATE", "_MARTS"]

prefixes = set()
for schema in df_all["TABLE_SCHEMA"].unique():
    for suffix in SCHEMA_SUFFIXES:
        if schema.endswith(suffix):
            prefix = schema[: -len(suffix)]
            if prefix:
                prefixes.add(prefix)
            break

raw_schemas = set(df_all[df_all["TABLE_SCHEMA"].str.endswith("_RAW")]["TABLE_SCHEMA"].unique())
prefixes    = sorted(p for p in prefixes if f"{p}_RAW" in raw_schemas)

print(f"Students with a _RAW schema: {len(prefixes)}")
for p in prefixes:
    print(f"  {p}")

In [ ]:
object_set = set(zip(df_all["TABLE_SCHEMA"], df_all["TABLE_NAME"]))

STAGES = [
    {
        "label":    "L1 — Seeds started",
        "lesson":   1,
        "required": lambda p: {
            (f"{p}_RAW", "CUSTOMERS"),
            (f"{p}_RAW", "ORDERS"),
        },
    },
    {
        "label":    "L1 — First model",
        "lesson":   1,
        "required": lambda p: {
            (f"{p}_STAGING", "STG_RAW__CUSTOMERS"),
        },
    },
    {
        "label":    "L3 — All staging",
        "lesson":   3,
        "required": lambda p: {
            (f"{p}_RAW",     "PRODUCTS"),
            (f"{p}_RAW",     "ORDER_ITEMS"),
            (f"{p}_RAW",     "PAYMENTS"),
            (f"{p}_STAGING", "STG_RAW__ORDERS"),
            (f"{p}_STAGING", "STG_RAW__PRODUCTS"),
            (f"{p}_STAGING", "STG_RAW__ORDER_ITEMS"),
            (f"{p}_STAGING", "STG_RAW__PAYMENTS"),
        },
    },
    {
        "label":    "L4 — Intermediate & Marts",
        "lesson":   4,
        "required": lambda p: {
            (f"{p}_INTERMEDIATE", "INT_ORDERS_WITH_PAYMENTS"),
            (f"{p}_INTERMEDIATE", "INT_ORDER_ITEMS_WITH_PRODUCTS"),
            (f"{p}_INTERMEDIATE", "INT_CUSTOMERS__ORDER_SUMMARY"),
            (f"{p}_MARTS",        "DIM_CUSTOMERS"),
            (f"{p}_MARTS",        "FCT_ORDERS"),
        },
    },
    {
        "label":    "L7 — Snapshot",
        "lesson":   7,
        "required": lambda p: {
            (f"{p}_SNAPSHOTS", "SNAP_ORDERS"),
        },
    },
    {
        "label":    "L12 — Incremental models",
        "lesson":   12,
        "required": lambda p: {
            (f"{p}_MARTS", "FCT_ORDERS_INCREMENTAL"),
        },
    },
]


def evaluate(prefix):
    return {
        stage["label"]: stage["required"](prefix).issubset(object_set)
        for stage in STAGES
    }


results = {p: evaluate(p) for p in prefixes}
print(f"Evaluated {len(results)} student(s)")

In [ ]:
STAGE_TO_LESSON = {
    "L1 — Seeds started":        1,
    "L1 — First model":          1,
    "L3 — All staging":          3,
    "L4 — Intermediate & Marts": 4,
    "L7 — Snapshot":             7,
    "L12 — Incremental models":  12,
}

df_summary = pd.DataFrame(results).T
df_summary.index.name = "Student"


def furthest_lesson(row):
    best = max((n for c, n in STAGE_TO_LESSON.items() if row.get(c, False)), default=0)
    return f">= L{best}" if best > 0 else "Not started"


df_summary.insert(0, "Furthest", df_summary.apply(furthest_lesson, axis=1))

df_display = df_summary.copy()
stage_cols  = [c for c in df_display.columns if c != "Furthest"]
df_display[stage_cols] = df_display[stage_cols].replace({True: "PASS", False: "----"})

lesson_order = {"Not started": -1, ">= L1": 1, ">= L3": 3, ">= L4": 4, ">= L7": 7, ">= L12": 12}
df_display["_sort"] = df_display["Furthest"].map(lambda x: lesson_order.get(x, 0))
df_display = df_display.sort_values("_sort", ascending=False).drop(columns="_sort")

print(f"{len(df_display)} student(s) found:")
display(df_display)

print("L5/L6/L8/L9/L10 leave no schema artifacts and cannot be detected from the database.")

In [ ]:
for prefix in df_display.index:
    student_df = df_all[df_all["TABLE_SCHEMA"].str.startswith(f"{prefix}_")].copy()
    furthest   = df_display.loc[prefix, "Furthest"]
    total      = len(student_df)

    print(f"\n{'─'*55}")
    print(f"  {prefix:<20} {furthest}   ({total} objects)")
    print(f"{'─'*55}")

    for schema, grp in student_df.groupby("TABLE_SCHEMA"):
        print(f"  {schema}")
        for _, r in grp.iterrows():
            type_char = "T" if "TABLE" in r["TABLE_TYPE"].upper() else "V"
            print(f"    . {r['TABLE_NAME']}  [{type_char}]")

In [ ]:
counts = df_display["Furthest"].value_counts().reindex(
    [">= L12", ">= L7", ">= L4", ">= L3", ">= L1", "Not started"]
).dropna().astype(int)

print("Students by furthest detected lesson:\n")
for level, count in counts.items():
    bar = "#" * count
    print(f"  {level:<15} {bar}  ({count})")